## Click and move

- Get an overview image

- user clicks on it 

- compute distance which stage moves using delta in pixels and pixel size

- move the stage

- check if position is correct --> move again

## x and y might have some angle to it --> so just move the x and y --> get angle with scan rotation
- x and y

In [ ]:
%matplotlib widget

import matplotlib.pyplot as plt


def get_haadf_image(microscope, client):
    data_key = microscope.acquire_scanned_image(["haadf"])
    image = client[data_key]["image"]["HAADF"].read()
    return image


def get_pixel_from_click(image):
    """Display image and return clicked pixel coordinates."""

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(image, cmap="gray", interpolation="none")
    ax.set_title("Click target pixel")
    ax.axis("off")

    clicked = {}

    def onclick(event):
        if event.xdata is None or event.ydata is None:
            return

        clicked["x"] = event.xdata
        clicked["y"] = event.ydata

        print(f"Clicked pixel: x={event.xdata:.1f}, y={event.ydata:.1f}")

        fig.canvas.mpl_disconnect(cid)

    cid = fig.canvas.mpl_connect("button_press_event", onclick)

    plt.show()

    return clicked

In [ ]:
# a. acquire HAADF image
image = get_haadf_image(microscope, client)

# b. click target pixel
clicked = get_pixel_from_click(image)

In [ ]:
x_click = clicked["x"]
y_click = clicked["y"]

print(x_click, y_click)

In [ ]:
def move_stage_to_pixel(stage, image, x_click, y_click, pixel_size):
    """Move stage so clicked pixel moves toward image center."""

    height, width = image.shape

    x_center = width / 2
    y_center = height / 2

    dx_pixels = x_click - x_center
    dy_pixels = y_click - y_center

    dx = dx_pixels * pixel_size
    dy = dy_pixels * pixel_size

    stage.x = stage.x + dx
    stage.y = stage.y + dy

    print(f"Moved x: {dx:.3e} m")
    print(f"Moved y: {dy:.3e} m")

In [ ]:
# c. user provides pixel size
pixel_size = 181e-12   # meters / pixel

# d. move stage
move_stage_to_pixel(
    stage,
    image,
    x_click,
    y_click,
    pixel_size,
)

In [ ]:
image_after = get_haadf_image(microscope, client)

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(image_after, cmap="gray", interpolation="none")
ax.set_title("HAADF after stage movement")
ax.axis("off")
plt.tight_layout()
plt.show()

One thing to experimentally verify is the sign convention. Depending on the microscope/stage coordinate system, one or both may need to be:
```python
stage.x = stage.x - dx
stage.y = stage.y - dy
```

## Try move to pixel from autoscript

In [ ]:
from autoscript_tem_microscope_client import TemMicroscopeClient
from autoscript_tem_microscope_client.structures import Point, MoveToPixelSettings, StemAcquisitionSettings
from autoscript_tem_microscope_client.enumerations import (
    DetectorType,
    CameraType,
    OptiStemMethod,
    OpticalMode,
    EdsDetectorType,
    ExposureTimeType,
    MoveToPixelMethod
)

In [ ]:
microscope = TemMicroscopeClient()

ip = '10.46.217.241'
port = 9095

# ip = '10.46.217.242'
# port = 9094

microscope.connect(ip, port)

In [ ]:
%matplotlib qt

import matplotlib.pyplot as plt


def get_pixel_from_click(image):
    result = {"x": None, "y": None}

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.imshow(image, cmap="gray")
    ax.set_title("Click a pixel")
    ax.axis("off")

    marker, = ax.plot([], [], "rx", markersize=12, markeredgewidth=2)

    def onclick(event):
        if event.inaxes != ax:
            return

        x = int(round(event.xdata))
        y = int(round(event.ydata))

        result["x"] = x
        result["y"] = y

        marker.set_data([x], [y])
        ax.set_title(f"Clicked: x={x}, y={y}")
        fig.canvas.draw()

        print(f"Clicked: x={x}, y={y}")

    fig.canvas.mpl_connect("button_press_event", onclick)

    plt.show(block=True)

    return result

In [ ]:
## collect an image

settings = StemAcquisitionSettings(
    1e-6,
    detector_types=[DetectorType.HAADF],
    size=1024,
)

image = microscope.acquisition.acquire_stem_images_advanced(settings)

In [ ]:
clicked = get_pixel_from_click(image[0].data)

In [ ]:
clicked
x = clicked['x']
y = clicked['y']

In [ ]:
# Set optical mode to STEM and diffraction
# microscope.optics.optical_mode = OpticalMode.STEM

# Set camera length to the largest value
# microscope.optics.camera_length.value = microscope.optics.camera_length.available_values[-1]

print(f"Stage position before: {microscope.specimen.stage.position}")
print(f"Beam shift before: {microscope.optics.deflectors.beam_shift}")





In [ ]:
# Move to the right by 20 pixels on the X axis (2068 - 4096 / 2 = 20 pixels)
microscope.specimen.stage.move_to_pixel(Point(x, y), DetectorType.HAADF,
    MoveToPixelSettings(method=MoveToPixelMethod.ANY))

In [ ]:
print(f"Stage position after: {microscope.specimen.stage.position}")
print(f"Beam shift after: {microscope.optics.deflectors.beam_shift}")

In [ ]:
## collect an image

settings = StemAcquisitionSettings(
    1e-6,
    detector_types=[DetectorType.HAADF],
    size=1024,
)

image = microscope.acquisition.acquire_stem_images_advanced(settings)

In [ ]:
microscope.optics.deflectors.beam_shift = [0, 0]

## Caliberation and matching
- caliberate the offset(scan rotation) in stage and the HAADF Detector


In [ ]:
# Imports and microscope helpers
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt

from matplotlib.widgets import RectangleSelector
from matplotlib.patches import Rectangle
from skimage.feature import match_template

# Stage coordinates assumed to be in meters.

def acquire_haadf():
    data_key = microscope.acquire_scanned_image(["haadf"])
    image = client[data_key]["image"]["HAADF"].read()
    return np.asarray(image).squeeze()


def read_stage_xy():
    """Read current absolute stage X,Y in meters."""
    return np.array([float(stage.x), float(stage.y)])


def move_stage_xy(target_xy):
    """Command absolute stage X,Y in meters."""
    # Replace this with your actual stage API if different.
    stage.x = float(target_xy[0])
    stage.y = float(target_xy[1])


def move_stage_relative(delta_xy):
    """Move relative to current position; return actual positions."""
    before = read_stage_xy()
    target = before + np.asarray(delta_xy, dtype=float)

    print("Requested move (nm):", (target - before) * 1e9)
    move_stage_xy(target)

    # Do not acquire while the stage is moving.
    # Replace this prompt with your API's wait_until_idle().
    input("Confirm stage motion has finished, then press Enter: ")

    after = read_stage_xy()
    print("Measured move (nm):", (after - before) * 1e9)

    return before, after


def image_center(image):
    h, w = image.shape
    return np.array([(w - 1) / 2, (h - 1) / 2])


In [ ]:
#Acquire image and select a feature

image0 = acquire_haadf()
stage0 = read_stage_xy()

selection = {}

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(image0, cmap="gray", origin="upper")
ax.set_title("Draw a rectangle around the feature")


def onselect(eclick, erelease):
    if eclick.xdata is None or erelease.xdata is None:
        return

    x1, x2 = sorted([int(eclick.xdata), int(erelease.xdata)])
    y1, y2 = sorted([int(eclick.ydata), int(erelease.ydata)])

    x1, x2 = np.clip([x1, x2], 0, image0.shape[1])
    y1, y2 = np.clip([y1, y2], 0, image0.shape[0])

    if x2 <= x1 or y2 <= y1:
        return

    selection["template"] = image0[y1:y2, x1:x2].copy()
    selection["pixel"] = np.array([
        x1 + (x2 - x1 - 1) / 2,
        y1 + (y2 - y1 - 1) / 2
    ])

    print("Selected feature:", selection["pixel"])


selector = RectangleSelector(
    ax, onselect,
    useblit=False,
    button=[1],
    interactive=True
)

plt.show()


In [ ]:

template = selection["template"]
pixel0 = selection["pixel"].copy()

print("Initial pixel:", pixel0)
print("Initial stage (m):", stage0)
print("Template shape:", template.shape)


In [ ]:
# Template matching function

def find_feature(image, template, min_score=0.5):
    image = np.asarray(image)
    template = np.asarray(template)

    result = match_template(image, template)

    iy, ix = np.unravel_index(np.argmax(result), result.shape)

    h, w = template.shape
    score = float(result[iy, ix])

    if not np.isfinite(score) or score < min_score:
        raise RuntimeError(
            f"Template match unreliable: score={score:.3f}"
        )

    pixel = np.array([
        ix + (w - 1) / 2,
        iy + (h - 1) / 2
    ])

    return pixel, score


def show_match(image, pixel, template, title):
    h, w = template.shape

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(image, cmap="gray")

    ax.add_patch(Rectangle(
        (pixel[0] - (w - 1) / 2 - 0.5,
         pixel[1] - (h - 1) / 2 - 0.5),
        w, h,
        edgecolor="red",
        facecolor="none",
        linewidth=2
    ))

    ax.plot(*pixel, "r+", markersize=12)
    ax.plot(*image_center(image), "g+", markersize=12)

    ax.set_title(title)
    plt.show()


In [ ]:
# First calibration: move stage along X

# Set safe calibration displacement for your microscope
step_x_nm = 10.0

_, stage1 = move_stage_relative(
    np.array([step_x_nm, 0.0]) * 1e-9
)

image1 = acquire_haadf()
pixel1, score1 = find_feature(image1, template)

show_match(
    image1, pixel1, template,
    f"After X move | score={score1:.3f}"
)

dp1 = pixel1 - pixel0
ds1 = stage1 - stage0

print("Pixel displacement:", dp1)
print("Stage displacement (nm):", ds1 * 1e9)


In [ ]:
# Second calibration: move stage along Y

step_y_nm = 10.0

_, stage2 = move_stage_relative(
    np.array([0.0, step_y_nm]) * 1e-9
)

image2 = acquire_haadf()
pixel2, score2 = find_feature(image2, template)

show_match(
    image2, pixel2, template,
    f"After Y move | score={score2:.3f}"
)

dp2 = pixel2 - pixel1
ds2 = stage2 - stage1

print("Pixel displacement:", dp2)
print("Stage displacement (nm):", ds2 * 1e9)


In [ ]:
# Calculate the transformation matrix A

# Matrix columns = measured displacement vectors
P = np.column_stack([dp1, dp2])
S = np.column_stack([ds1, ds2])

# Prevent calibration from using degenerate movements
if np.linalg.cond(P) > 100:
    raise RuntimeError(
        "Pixel displacement matrix is poorly conditioned. "
        "Try larger, independent calibration movements."
    )

# Pixel displacement -> stage displacement
A = S @ np.linalg.inv(P)

print("A (meters/pixel):")
print(A)

print("\nA (nm/pixel):")
print(A * 1e9)

# Extract nearest orthogonal orientation matrix
U, scales, Vt = np.linalg.svd(A)
R = U @ Vt

print("\nOrientation matrix R:")
print(R)

print("\nScale along principal axes (nm/pixel):")
print(scales * 1e9)

if np.linalg.det(R) > 0:
    angle = np.degrees(np.arctan2(R[1, 0], R[0, 0]))
    print(f"\nRotation angle: {angle:.2f} degrees")
else:
    print("\nOrientation contains a reflection (axis inversion).")


In [ ]:
# Center the feature using the calibrated matrix

def move_feature_to_center(image, feature_pixel, A):
    center = image_center(image)

    # Where the feature needs to move in IMAGE coordinates
    pixel_displacement = center - np.asarray(feature_pixel)

    # Required physical STAGE movement in meters
    stage_displacement = A @ pixel_displacement

    print("Current feature pixel:", feature_pixel)
    print("Target image center:", center)
    print("Required pixel shift:", pixel_displacement)
    print("Required stage shift (nm):", stage_displacement * 1e9)

    return move_stage_relative(stage_displacement)


# Center the feature identified after calibration
stage_before_center, stage_after_center = move_feature_to_center(
    image=image2,
    feature_pixel=pixel2,
    A=A
)


In [ ]:
# Reuse the calibration for future clicks
# Acquire fresh image
image_new = acquire_haadf()

# Draw an ROI using the same selection procedure
# to obtain the new feature's pixel center

new_feature_pixel = np.array([350, 190])  # Example

# Move selected feature to the image center
move_feature_to_center(
    image_new,
    new_feature_pixel,
    A
)

In [ ]:
# Recommended improvement: iterative centering

def center_feature_closed_loop(
    image,
    template,
    A,
    tolerance_pixels=3,
    max_iterations=3,
):
    for iteration in range(max_iterations):
        feature_pixel, score = find_feature(image, template)

        center = image_center(image)
        error = center - feature_pixel

        print(
            f"Iteration {iteration + 1}: "
            f"centering error = {np.linalg.norm(error):.2f} px"
        )

        if np.linalg.norm(error) <= tolerance_pixels:
            print("Feature successfully centered.")
            return image

        stage_offset = A @ error

        # Check that the requested movement is safe before execution
        move_stage_relative(stage_offset)

        image = acquire_haadf()

    print("Maximum correction iterations reached.")
    return image


In [ ]:
# Call it after calibration:
image_centered = center_feature_closed_loop(
    image=image2,
    template=template,
    A=A,
    tolerance_pixels=3
)

### Using just the theta formulation to get rotation matrix

- Acquire HAADF image.

- User draws a box around a feature.
- Extract the template and record initial stage position.
- Move the stage.
- Acquire another HAADF image and find the feature using match_template.
- Calculate the rotation matrix and scale.
- Predict the stage position needed to bring the feature to the image center.
- Autoscript commands needed? -> get_image, get_stage_postions, set_stage_positon


In [ ]:
import os
import json
import tango
import numpy as np
from pprint import pprint
import matplotlib.pyplot as plt
from tiled.client import from_uri

%matplotlib ipympl

DB_HOST = "10.46.217.241"
# DB_HOST = "localhost"
DB_PORT = 9094

os.environ["TANGO_HOST"] = f"{DB_HOST}:{DB_PORT}"

server_names = ['stage', 'scan', 'eds', 'camera', 'data', 'instrument']

scan = tango.DeviceProxy("asyncroscopy/scan/default")
camera = tango.DeviceProxy("asyncroscopy/camera/default")
microscope = tango.DeviceProxy("asyncroscopy/instrument/default")
data = tango.DeviceProxy("asyncroscopy/data/default")
stage = tango.DeviceProxy("asyncroscopy/stage/default")


for proxy in [scan, microscope, data]:
    print(proxy)
    proxy.set_timeout_millis(120_000)
    proxy.ping()
    print(proxy.name(), proxy.state())
    
    
config = json.loads(data.get_config())
client = from_uri(config.get("uri"))
print("Tiled keys:", list(client))

client = from_uri(config.get("uri"))
print("Tiled keys:", list(client))

In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import RectangleSelector
from matplotlib.patches import Rectangle
from skimage.feature import match_template


# ============================================================
# 1. Microscope functions
# ============================================================

def acquire_haadf():
    data_key = microscope.acquire_scanned_image(["haadf"])
    return np.asarray(
        client[data_key]["image"]["HAADF"].read()
    ).squeeze()


def get_stage_position():
    # Modify according to your stage API
    return np.array([stage.x, stage.y], dtype=float)


def move_stage(dx, dy):
    # Relative movement; adjust for your stage API.
    current = get_stage_position()

    stage.x = float(current[0] + dx)
    stage.y = float(current[1] + dy)

    input("Confirm stage movement has finished, then press Enter.")


# ============================================================
# 2. Acquire initial image and select template
# ============================================================

image_before = acquire_haadf()
stage_before = get_stage_position()

selection = {}

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(image_before, cmap="gray", origin="upper")
ax.set_title("Draw a box around the feature")


def onselect(eclick, erelease):

    if eclick.xdata is None or erelease.xdata is None:
        return

    x1, x2 = sorted([int(eclick.xdata), int(erelease.xdata)])
    y1, y2 = sorted([int(eclick.ydata), int(erelease.ydata)])

    x1, x2 = np.clip([x1, x2], 0, image_before.shape[1])
    y1, y2 = np.clip([y1, y2], 0, image_before.shape[0])

    if x2 <= x1 or y2 <= y1:
        return

    selection["template"] = image_before[y1:y2, x1:x2].copy()
    selection["pixel"] = np.array([
        (x1 + x2 - 1) / 2,
        (y1 + y2 - 1) / 2
    ])

    print("Selected feature:", selection["pixel"])


selector = RectangleSelector(
    ax,
    onselect,
    useblit=False,
    button=[1],
    interactive=True
)

plt.show()

In [ ]:
# %matplotlib widget

# import matplotlib
# import ipympl
# import ipywidgets
# import matplotlib.pyplot as plt
# import numpy as np

# print("Matplotlib:", matplotlib.__version__)
# print("ipympl:", ipympl.__version__)
# print("ipywidgets:", ipywidgets.__version__)
# print("Backend:", matplotlib.get_backend())

# fig, ax = plt.subplots(figsize=(6, 6))
# ax.imshow(np.random.rand(512, 512), cmap="gray")
# plt.show()

In [ ]:
# Move stage, reacquire and match feature
template = selection["template"]
pixel_before = selection["pixel"]

# Stage displacement in meters (adjust for your FOV)
dx = 10*1e-9
dy = 0.0

move_stage(dx, dy)

stage_after = get_stage_position()
image_after = acquire_haadf()

# Template matching
result = match_template(image_after, template)

iy, ix = np.unravel_index(np.argmax(result), result.shape)
h, w = template.shape

pixel_after = np.array([
    ix + (w - 1) / 2,
    iy + (h - 1) / 2
])

# Displacements
dP = pixel_after - pixel_before
dS = stage_after - stage_before

print("Detector displacement (pixels):", dP)
print("Stage displacement (meters):", dS)
print("Match score:", result.max())

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(image_before, cmap="gray")
axes[0].plot(*pixel_before, "r+", markersize=12)
axes[0].set_title("Before stage movement")

axes[1].imshow(image_after, cmap="gray")
axes[1].add_patch(Rectangle(
    (ix - 0.5, iy - 0.5),
    w, h,
    edgecolor="red",
    facecolor="none",
    linewidth=2
))
axes[1].plot(*pixel_after, "r+", markersize=12)
axes[1].set_title("After stage movement")

plt.tight_layout()
plt.show()

In [ ]:
# Calculate rotation matrix

if np.linalg.norm(dS) == 0 or np.linalg.norm(dP) == 0:
    raise ValueError("Stage and detector displacements must be nonzero.")

theta = (
    np.arctan2(dP[1], dP[0])
    - np.arctan2(dS[1], dS[0])
)

scale = np.linalg.norm(dP) / np.linalg.norm(dS)

R = np.array([
    [np.cos(theta), -np.sin(theta)],
    [np.sin(theta),  np.cos(theta)]
])

B = scale * R

print(f"Rotation angle: {np.degrees(theta):.2f} degrees")
print("Rotation matrix:\n", R)
print("Stage-to-detector matrix:\n", B)
print(f"Scale: {scale:.3e} pixels/meter")

In [ ]:
# Predict and move the feature to the image center
def calculate_stage_target(pixel, stage_position, image, B):
    height, width = image.shape

    center = np.array([
        (width - 1) / 2,
        (height - 1) / 2
    ])

    pixel_shift = center - pixel

    # Stage shift required to produce pixel_shift
    stage_shift = np.linalg.solve(B, pixel_shift)
    #stage_shift = np.dot(pixel_shift, B) maybe test with pixel_shift @ B

    return stage_position + stage_shift


stage_target = calculate_stage_target(
    pixel=pixel_after,
    stage_position=stage_after,
    image=image_after,
    B=B
)

print("Current stage:", stage_after)
print("Target stage:", stage_target)
print("Required movement (nm):", (stage_target - stage_after) * 1e9)

In [ ]:
# Review the predicted movement before running this cell.
delta = stage_target - get_stage_position()

move_stage(*delta)

image_final = acquire_haadf()

result = match_template(image_final, template)
iy, ix = np.unravel_index(np.argmax(result), result.shape)

h, w = template.shape

pixel_final = np.array([
    ix + (w - 1) / 2,
    iy + (h - 1) / 2
])

center = np.array([
    (image_final.shape[1] - 1) / 2,
    (image_final.shape[0] - 1) / 2
])

error = np.linalg.norm(pixel_final - center)

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(image_final, cmap="gray")
ax.add_patch(Rectangle(
    (ix - 0.5, iy - 0.5),
    w, h,
    edgecolor="red",
    facecolor="none",
    linewidth=2
))
ax.plot(*pixel_final, "r+", markersize=14, label="Matched feature")
ax.plot(*center, "g+", markersize=14, label="Image center")
ax.legend()
ax.set_title(f"After centering | Error: {error:.2f} pixels")
plt.show()

print("Final feature:", pixel_final)
print("Centering error (pixels):", error)